# BioDYM Material Flow Analysis - Scientific Notebook

A streamlined notebook for Material Flow Analysis using the BioDYM framework with enhanced plotting capabilities.

## Workflow Overview

This notebook follows a structured approach to Material Flow Analysis:

1. **Setup and Data Loading** - Prepare environment and load input data
2. **Calculation & Validation** - Execute MFA analysis and verify results
3. **Visualization** - Comprehensive analysis and exploration
4. **Export** - Save results and generate documentation

---

# 1. Setup and Data Loading

This section prepares the analysis environment and loads the input data.

## 1.1 Environment Setup

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from IPython.display import display, HTML, Markdown

In [ ]:
# Add BioDYM modules to path
src_path = os.path.join(os.getcwd(), 'src')
sys.path.insert(0, src_path)

In [ ]:
# Add ODYM framework to path
biodym_mfa_tool_dir = os.getcwd()
odym_path = os.path.join(
    biodym_mfa_tool_dir, "framework", "ODYM-master_20241127", "odym", "modules"
)
sys.path.insert(0, odym_path)

In [ ]:
# Add bioDYM add-on to path
biodym_addon_path = os.path.join(
    biodym_mfa_tool_dir, "framework", "bioDYM_add-on", "modules"
)
sys.path.insert(0, biodym_addon_path)

In [ ]:
# Import BioDYM modules
try:
    import config
    import data_loader
    import system_setup
    import utils
    from engine import solver
    import plotting
    print("✅ BioDYM modules imported successfully")
except ImportError as e:
    print(f"❌ Import error: {e}")
    raise

In [ ]:
# Set up plotting
plt.style.use('default')
print("📊 Plotting environment ready")

## 1.2 Data Input Configuration

**Change this variable to your Excel file:**

In [ ]:
input_file = "data/01_input/250707_Template_CS1.xlsx"

In [ ]:
print(f"📁 Input file: {input_file}")

## 1.3 Data Loading and Validation

In [ ]:
print("\n" + "="*60)
print("📊 LOADING AND VALIDATING DATA")
print("="*60)

In [ ]:
# Load Excel file
try:
    input_data = pd.read_excel(
        input_file,
        sheet_name=None,
        header=0,
        engine='openpyxl',
        na_values=['N.A.', 'NA', 'n/a']
    )
    print(f"✅ Excel file loaded: {len(input_data)} sheets")
except Exception as e:
    print(f"❌ Error loading file: {e}")
    raise

In [ ]:
# Display sheet overview
print("\n📋 Sheet Overview:")
for sheet_name, df in input_data.items():
    print(f"   {sheet_name}: {df.shape[0]} rows × {df.shape[1]} columns")

In [ ]:
# Validate required sheets
required_sheets = [
    '1_1_Definition_Flows',
    '1_2_Data_Flows', 
    '2_1_Definition_Processes',
    '2_4_Initial_Stock',  # Correct sheet name
    '2_5_dynamic_tcs'
]

In [ ]:
missing_sheets = [sheet for sheet in required_sheets if sheet not in input_data.keys()]
if missing_sheets:
    print(f"\n⚠️ Missing required sheets: {missing_sheets}")
else:
    print("\n✅ All required sheets present")

## 1.4 System Configuration Extraction

In [ ]:
print("\n" + "="*60)
print("⚙️ EXTRACTING CONFIGURATION")
print("="*60)

In [ ]:
# Extract time range from flow data
flow_data = input_data['1_2_Data_Flows']
years = sorted(flow_data['Year_Flow'].unique())
start_year = int(min(years))
end_year = int(max(years))

In [ ]:
print(f"📅 Time range: {start_year} - {end_year}")

In [ ]:
# Extract elements from flow data
elements = ['material', 'WC', 'DM', 'CC']  # Default elements
print(f"🧪 Elements: {elements}")

In [ ]:
# Check for Monte Carlo parameters
has_mc = '4_1_Uncertainty_Parameters' in input_data.keys()
print(f"🎲 Monte Carlo available: {'Yes' if has_mc else 'No'}")

In [ ]:
# Check for DSM parameters
has_dsm = '3_1_Definition_DSM' in input_data.keys()
print(f"📈 DSM available: {'Yes' if has_dsm else 'No'}")

In [ ]:
# Check for FOMP parameters
has_fomp = '3_2_Definition_FOMP' in input_data.keys()
print(f"🌱 FOMP available: {'Yes' if has_fomp else 'No'}")

## 1.5 Configuration Review

In [ ]:
print("\n" + "="*60)
print("✅ CONFIGURATION CONFIRMATION")
print("="*60)

In [ ]:
config_summary = f"""
**Analysis Configuration:**
- Input File: {input_file}
- Time Range: {start_year} - {end_year}
- Elements: {', '.join(elements)}
- Monte Carlo: {'Enabled' if has_mc else 'Disabled'}
- DSM: {'Enabled' if has_dsm else 'Disabled'}
- FOMP: {'Enabled' if has_fomp else 'Disabled'}
"""

In [ ]:
display(Markdown(config_summary))

# 2. Calculation & Validation

This section executes the MFA calculation and immediately validates the results through mass balance checks.

## 2.1 Model Initialization

In [ ]:
print("\n" + "="*60)
print("🚀 RUNNING MFA CALCULATION")
print("="*60)

In [ ]:
# 1. Setup model scope
print("📋 Setting up model scope...")
try:
    model_classification, index_table = system_setup.define_model_scope(
        start_year, end_year, elements
    )
    print("✅ Model scope defined")
except Exception as e:
    print(f"❌ Error setting up model scope: {e}")
    raise

In [ ]:
# 2. Initialize MFA system
print("🔧 Initializing MFA system...")
try:
    mfa_system_base = system_setup.initialize_mfa_system(
        model_classification, index_table
    )
    print("✅ MFA system initialized")
except Exception as e:
    print(f"❌ Error initializing MFA system: {e}")
    raise

In [ ]:
# 3. Load and define processes
print("📊 Loading processes and data...")
try:
    mfa_system_base, all_excel_data = system_setup.load_and_define_processes(
        mfa_system_base, input_file, data_loader
    )
    print("✅ Processes and data loaded")
except Exception as e:
    print(f"❌ Error loading processes: {e}")
    raise

In [ ]:
# 4. Load parameters
print("⚙️ Loading parameters...")
try:
    dsm_params = data_loader.load_dsm_parameters(all_excel_data)
    fomp_params = data_loader.load_fomp_parameters(all_excel_data)
    uncertainty_params = data_loader.load_uncertainty_definitions(all_excel_data)
    print("✅ Parameters loaded")
except Exception as e:
    print(f"❌ Error loading parameters: {e}")
    raise

## 2.2 MFA Calculation Execution

In [ ]:
# 5. Define flows and parameters
print("🔗 Defining flows and parameters...")
try:
    mfa_system_configured, _ = system_setup.define_flows_and_parameters(
        mfa_system_base, all_excel_data
    )
    print(f"✅ System configured: {len(mfa_system_configured.ProcessList)} processes, "
          f"{len(mfa_system_configured.FlowDict)} flows, {len(mfa_system_configured.StockDict)} stocks")
except Exception as e:
    print(f"❌ Error defining flows and parameters: {e}")
    raise

In [ ]:
# 6. Run calculation
print("🧮 Running calculation...")
try:
    mfa_system_with_results, dsm_details = solver.run_mfa_calculation(
        mfa_system_configured, dsm_params, fomp_params, config
    )
    print("✅ Calculation completed successfully!")
except Exception as e:
    print(f"❌ Calculation error: {e}")
    import traceback
    traceback.print_exc()
    raise

## 2.3 Mass Balance Validation

In [ ]:
print("\n" + "="*60)
print("⚖️ MASS BALANCE VERIFICATION")
print("="*60)

In [ ]:
# Calculate mass balance errors
mass_balance_errors = []
for process in mfa_system_with_results.ProcessList:
    if hasattr(process, 'MassBalance') and process.MassBalance is not None:
        for year_idx, year in enumerate(range(start_year, end_year + 1)):
            for element_idx, element in enumerate(elements):
                error = process.MassBalance[year_idx, element_idx]
                if abs(error) > 1e-6:  # Significant error threshold
                    mass_balance_errors.append({
                        'Process': process.Name,
                        'Year': year,
                        'Element': element,
                        'Error': error
                    })

In [ ]:
if mass_balance_errors:
    print("⚠️ Mass balance errors detected:")
    error_df = pd.DataFrame(mass_balance_errors)
    display(error_df)
else:
    print("✅ All mass balances within acceptable limits")

In [ ]:
# Mass Balance Error Visualization
print("\n⚖️ Creating optimized mass balance error plots...")
try:
    # Use the optimized mass balance error function
    plotting.plot_optimized_mass_balance_error(mfa_system_with_results)
    print("✅ Optimized mass balance error plots created")
    print("   🚀 Performance: Pre-calculated flow sums, memory optimized")
    print("   🎨 Visualization: Color-coded errors (red=created, green=destroyed)")
    print("   📁 Export: Enhanced export options (PNG, PDF, SVG, HTML)")
except Exception as e:
    print(f"⚠️ Could not create mass balance error plots: {e}")

## 2.4 Results Overview

In [ ]:
print("\n" + "="*60)
print("📈 RESULTS OVERVIEW")
print("="*60)

In [ ]:
# Display final stock values
print("\n📊 Final Stock Values (Year {end_year}):")
final_stocks = []
for stock_name, stock in mfa_system_with_results.StockDict.items():
    if stock_name.startswith('S_'):  # Absolute stocks only
        final_value = stock.Values[-1, 0]  # Material dimension, final year
        final_stocks.append({
            'Stock': stock_name,
            'Final Value (Mg)': final_value
        })

In [ ]:
if final_stocks:
    stocks_df = pd.DataFrame(final_stocks)
    display(stocks_df)

In [ ]:
# Display flow summary
print("\n🔄 Flow Summary:")
flow_summary = []
for flow_id, flow in mfa_system_with_results.FlowDict.items():
    avg_flow = np.mean(flow.Values[:, 0])  # Average material flow
    flow_summary.append({
        'Flow ID': flow_id,
        'From': flow.P_Start,
        'To': flow.P_End,
        'Avg Flow (Mg/year)': avg_flow
    })

In [ ]:
if flow_summary:
    flows_df = pd.DataFrame(flow_summary)
    display(flows_df.head(10))  # Show first 10 flows

# 3. Visualization

This section provides comprehensive analysis and exploration through various visualization tools.

In [ ]:
print("\n" + "="*60)
print("📊 VISUALIZATION")
print("="*60)

## 3.1 System Overview

In [ ]:
print("\n" + "-"*40)
print("3.1 SYSTEM OVERVIEW")
print("-"*40)

### 3.1.1 Material Flow Sankey Diagram

In [ ]:
print("🔗 Creating interactive Sankey diagram...")
try:
    # Use the enhanced interactive Sankey function with DSM/FOMP parameters
    plotting.plot_interactive_sankey(mfa_system_with_results, dsm_params, fomp_params)
    print("✅ Interactive Sankey diagram created")
    print("   📊 Features: Multi-process selection, color coding, export options")
    print("   🎨 Process types: Regular (blue), DSM (orange), FOMP (green)")
    print("   📁 Export: PNG with timestamped filenames in organized folders")
except Exception as e:
    print(f"⚠️ Could not create interactive Sankey diagram: {e}")
    import traceback
    traceback.print_exc()

### 3.1.2 Stock Bar Chart

In [ ]:
print("\n📊 Creating stock bar chart...")
try:
    # Use the new simple stock bar chart function
    plotting.plot_stock_bars_simple(mfa_system_with_results, dsm_params, fomp_params)
    print("✅ Stock bar chart created")
    print("   📊 Features: Multi-process selection, element selection, year slider")
    print("   🎨 Color coding: Regular (blue), DSM (orange), FOMP (green)")
    print("   📈 Interactive: Real-time updates with widget controls")
except Exception as e:
    print(f"⚠️ Could not create stock bar chart: {e}")

### 3.1.3 Individual Process Analysis

In [ ]:
print("\n📊 Creating individual process analysis...")
try:
    # Use the new individual process analysis function
    plotting.plot_individual_process_analysis(mfa_system_with_results, dsm_params, fomp_params)
    print("✅ Individual process analysis created")
    print("   📊 Features: 3-panel layout (Input | Stock | Outflow)")
    print("   🎛️ Controls: Process selection, element selection")
    print("   🎨 Color coding: Regular (blue), DSM (orange), FOMP (green)")
except Exception as e:
    print(f"⚠️ Could not create individual process analysis: {e}")

## 3.2 Individual Process Analysis

In [ ]:
print("\n" + "-"*40)
print("3.2 INDIVIDUAL PROCESS ANALYSIS")
print("-"*40)

### 3.2.1 DSM Process Analysis

In [ ]:
print("\n📈 3.2.1 DSM Process Analysis:")
try:
    if has_dsm and dsm_details:
        plotting.plot_dsm_stock_details(mfa_system_with_results, dsm_params, dsm_details)
        print("✅ DSM process analysis plots created")
    else:
        print("ℹ️ No DSM processes available")
except Exception as e:
    print(f"⚠️ Could not create DSM process analysis: {e}")

### 3.2.2 FOMP Process Analysis

In [ ]:
print("\n🌱 3.2.2 FOMP Process Analysis:")
try:
    if has_fomp and fomp_params:
        plotting.plot_fomp_stock_details(mfa_system_with_results, fomp_params)
        print("✅ FOMP process analysis plots created")
    else:
        print("ℹ️ No FOMP processes available")
except Exception as e:
    print(f"⚠️ Could not create FOMP process analysis: {e}")

## 3.3 Detailed Component Analysis

In [ ]:
print("\n" + "-"*40)
print("3.3 DETAILED COMPONENT ANALYSIS")
print("-"*40)

### 3.3.1 Individual Stock Analysis

In [ ]:
print("📊 Creating individual stock analysis...")
try:
    plotting.plot_individual_stocks(mfa_system_with_results, dsm_params, fomp_params)
    print("✅ Individual stock analysis created")
    print("   🎨 Features: Process type color coding, delta stock visualization")
    print("   📊 Options: Multi-stock selection, bar/line charts, cumulative values")
except Exception as e:
    print(f"⚠️ Could not create individual stock analysis: {e}")

### 3.3.2 Individual Flow Analysis

In [ ]:
print("\n🔄 Creating individual flow analysis...")
try:
    plotting.plot_individual_flows(mfa_system_with_results)
    print("✅ Individual flow analysis created")
    print("   📊 Features: Multi-flow selection, cumulative vs. individual values")
    print("   📈 Options: Bar/line charts, element-specific analysis")
except Exception as e:
    print(f"⚠️ Could not create individual flow analysis: {e}")

### 3.3.3 System Efficiency Metrics

In [ ]:
print("\n📈 Creating system efficiency metrics...")
try:
    plotting.plot_system_efficiency_metrics(mfa_system_with_results)
    print("✅ System efficiency metrics created")
except Exception as e:
    print(f"⚠️ Could not create system efficiency metrics: {e}")

## 3.4 Advanced Analysis

In [ ]:
print("\n" + "-"*40)
print("3.4 ADVANCED ANALYSIS")
print("-"*40)

### 3.4.1 Summary Dashboard

In [ ]:
print("📊 Creating summary dashboard...")
try:
    plotting.plot_summary_dashboard(mfa_system_with_results, dsm_params, fomp_params)
    print("✅ Summary dashboard created")
except Exception as e:
    print(f"⚠️ Could not create summary dashboard: {e}")

### 3.4.2 Monte Carlo Uncertainty Analysis

In [ ]:
print("\n🎲 Creating integrated Monte Carlo dashboard...")
try:
    # Create sample MC results for demonstration (replace with actual MC data)
    if has_mc:
        # Generate sample MC results for demonstration
        n_iterations = 100
        mc_results = pd.DataFrame({
            'iteration': range(n_iterations),
            'Total_Stock_material': np.random.normal(924.6, 50, n_iterations),
            'Total_Stock_WC': np.random.normal(0, 5, n_iterations),
            'Total_Stock_DM': np.random.normal(0, 5, n_iterations),
            'Total_Stock_CC': np.random.normal(0, 2, n_iterations),
            'parameter_1': np.random.uniform(0.8, 1.2, n_iterations),
            'parameter_2': np.random.uniform(0.9, 1.1, n_iterations)
        })
        
        # Use the new integrated MC dashboard
        plotting.plot_monte_carlo_integrated_dashboard(
            mfa_system_with_results, mc_results, dsm_params, fomp_params
        )
        print("✅ Integrated Monte Carlo dashboard created")
        print("   📊 4-Panel Layout: Deterministic vs MC, Distribution, Sensitivity, Confidence")
        print("   🎯 Features: Real-time updates, confidence intervals, error bands")
        print("   📈 Analysis: Parameter sensitivity, correlation matrices")
    else:
        print("ℹ️ Monte Carlo analysis not available (no uncertainty parameters)")
        print("   To enable MC analysis, add uncertainty parameters to your input file.")
except Exception as e:
    print(f"⚠️ Could not create Monte Carlo dashboard: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
# Individual MC plots
print("\n📊 Creating individual Monte Carlo plots...")
try:
    if has_mc and 'mc_results' in locals():
        # Individual MC plots using existing functions
        plotting.plot_mc_distribution(mc_results, 'Total_Stock_material', 'Mg', 'Material Stock Distribution')
        plotting.plot_mc_correlation_matrix(mc_results, title='MC Parameter Correlations')
        plotting.plot_mc_confidence_intervals(mc_results, 'Total_Stock_material', unit='Mg')
        print("✅ Individual Monte Carlo plots created")
        print("   📊 Distribution: Histogram and box plot analysis")
        print("   🔗 Correlation: Parameter relationship matrix")
        print("   📈 Confidence: Percentile-based uncertainty analysis")
    else:
        print("ℹ️ No MC results available for individual plots")
except Exception as e:
    print(f"⚠️ Could not create individual MC plots: {e}")

# 4. Export

This section saves results and generates documentation for the analysis.

In [ ]:
print("\n" + "="*60)
print("💾 EXPORTING RESULTS")
print("="*60)

## 4.1 Results Export

In [ ]:
# Export to Excel
output_file = "data/02_output/results_scientific.xlsx"
try:
    utils.export_results_to_excel(mfa_system_with_results, output_file)
    print(f"✅ Results exported to: {output_file}")
except Exception as e:
    print(f"⚠️ Export error: {e}")

## 4.2 Configuration Export

In [ ]:
# Export configuration summary
config_file = output_file.replace('.xlsx', '_config.xlsx')
try:
    config_summary = pd.DataFrame([{
        'Input File': input_file,
        'Start Year': start_year,
        'End Year': end_year,
        'Elements': ', '.join(elements),
        'Monte Carlo': has_mc,
        'DSM': has_dsm,
        'FOMP': has_fomp
    }])
    config_summary.to_excel(config_file, index=False)
    print(f"✅ Configuration exported to: {config_file}")
except Exception as e:
    print(f"⚠️ Config export error: {e}")

## 4.3 Analysis Summary

In [ ]:
print("\n" + "="*60)
print("🎉 ANALYSIS COMPLETE")
print("="*60)

In [ ]:
summary = f"""
**Analysis Summary:**
- ✅ Input file processed successfully
- ✅ Configuration extracted automatically
- ✅ MFA calculation completed
- ✅ Mass balance verified
- ✅ Visualizations generated
- ✅ Results exported

**Key Results:**
- Time period: {start_year} - {end_year}
- Processes analyzed: {len(mfa_system_with_results.ProcessList)}
- Flows tracked: {len(mfa_system_with_results.FlowDict)}
- Stocks modeled: {len(mfa_system_with_results.StockDict)}
- Mass balance errors: {len(mass_balance_errors)}

**Files Generated:**
- Main results: {output_file}
- Configuration: {config_file}
"""

In [ ]:
display(Markdown(summary))

In [ ]:
print("\n📊 Analysis completed successfully!") 